# Snowflake Online Store → Triton Inference Demo

This notebook demonstrates the end-to-end flow of using **Snowflake as a Feast online store** to serve features for model inference.

It connects to the **existing `ani-featurestore-snowflake`** FeatureStore CR running in the cluster — no direct Snowflake credentials needed.

## Architecture
```
Notebook
   ↓  (Feast remote online store)
ani-featurestore-snowflake online server   ← existing FeatureStore CR
   ↓
Snowflake: ANI_FEAST_DEMO.FEAST_ONLINE
   ↓  (features retrieved)
Triton inference endpoint
   ↓
Anomaly score
```

## Step 1 — Install Dependencies

In [ ]:
!pip install --quiet 'feast' pandas requests

## Step 2 — Configuration

These point to the **existing `ani-featurestore-snowflake`** services created by the Feast operator.  
No Snowflake credentials are needed here — the online server handles that internally.

In [ ]:
import os

# Feast operator creates these services for ani-featurestore-snowflake
FEAST_ONLINE_HOST    = os.environ.get(
    "FEAST_SNOWFLAKE_ONLINE_HOST",
    "https://feast-ani-featurestore-snowflake-online.ani-datascience.svc.cluster.local:443",
)
FEAST_REGISTRY_HOST  = os.environ.get(
    "FEAST_SNOWFLAKE_REGISTRY_HOST",
    "feast-ani-featurestore-snowflake-registry.ani-datascience.svc.cluster.local:443",
)
CA_CERT = "/var/run/secrets/kubernetes.io/serviceaccount/service-ca.crt"

TRITON_ENDPOINT = os.environ.get(
    "TRITON_ENDPOINT",
    "http://ani-predictive-fs-predictor.ani-demo-lab.svc.cluster.local:8080",
)
MODEL_NAME = os.environ.get("MODEL_NAME", "ani-predictive-fs")

# Change this to try different scenarios
SCENARIO       = "registration_storm"   # options: normal | registration_storm | malformed_invite
DEMO_WINDOW_ID = f"demo-{SCENARIO}-001"

NUMERIC_FEATURES = [
    "register_rate", "invite_rate", "bye_rate",
    "error_4xx_ratio", "error_5xx_ratio", "latency_p95",
    "retransmission_count", "inter_arrival_mean", "payload_variance",
]

print(f"Feast online store  : {FEAST_ONLINE_HOST}")
print(f"Feast registry      : {FEAST_REGISTRY_HOST}")
print(f"Triton endpoint     : {TRITON_ENDPOINT}")
print(f"Scenario            : {SCENARIO}")
print(f"Demo window_id      : {DEMO_WINDOW_ID}")

## Step 3 — Feature Scenarios

The 9 SIP network features the model was trained on.

In [ ]:
SCENARIOS = {
    "normal": {
        "register_rate":        0.13,   # calm — ~1 REGISTER per 8 seconds
        "invite_rate":          0.03,
        "bye_rate":             0.03,
        "error_4xx_ratio":      0.0,
        "error_5xx_ratio":      0.0,
        "latency_p95":          35.0,
        "retransmission_count": 0.0,
        "inter_arrival_mean":   7.5,
        "payload_variance":     40.0,
    },
    "registration_storm": {
        "register_rate":        6.0,    # 46x normal — flood of REGISTER messages
        "invite_rate":          0.2,
        "bye_rate":             0.1,
        "error_4xx_ratio":      0.05,
        "error_5xx_ratio":      0.02,
        "latency_p95":          180.0,  # slow — system under stress
        "retransmission_count": 12.0,   # many retries
        "inter_arrival_mean":   0.8,
        "payload_variance":     60.0,
    },
    "malformed_invite": {
        "register_rate":        0.2,
        "invite_rate":          1.6,
        "bye_rate":             0.1,
        "error_4xx_ratio":      0.62,   # 62% of messages are errors
        "error_5xx_ratio":      0.0,
        "latency_p95":          440.0,
        "retransmission_count": 8.0,
        "inter_arrival_mean":   0.9,
        "payload_variance":     120.0,
    },
}

features = SCENARIOS[SCENARIO]
print(f"Feature values for scenario '{SCENARIO}':")
for k, v in features.items():
    print(f"  {k:<25} {v}")

## Step 4 — Connect Feast to `ani-featurestore-snowflake`

The `remote` online store type tells Feast to talk to the existing **`ani-featurestore-snowflake` online store server**  
rather than connecting to Snowflake directly. The server handles all Snowflake communication internally.

In [ ]:
import tempfile, textwrap
from datetime import timedelta
from feast import Entity, FeatureView, Field, FeatureStore, FileSource
from feast.types import Float32
from feast.value_type import ValueType
from feast.data_format import ParquetFormat

tmpdir = tempfile.mkdtemp()

with open(f"{tmpdir}/feature_store.yaml", "w") as f:
    f.write(textwrap.dedent(f"""
        project: ani_anomaly_featurestore
        provider: local
        online_store:
          type: remote
          path: {FEAST_ONLINE_HOST}
          cert: {CA_CERT}
        registry:
          registry_type: remote
          path: {FEAST_REGISTRY_HOST}
          cert: {CA_CERT}
        entity_key_serialization_version: 3
    """))

feature_window = Entity(
    name="feature_window",
    join_keys=["window_id"],
    value_type=ValueType.STRING,
)
ani_window_numeric_v1 = FeatureView(
    name="ani_window_numeric_v1",
    entities=[feature_window],
    ttl=timedelta(days=3650),
    schema=[Field(name=f, dtype=Float32) for f in NUMERIC_FEATURES],
    source=FileSource(
        path="/tmp/dummy.parquet",
        file_format=ParquetFormat(),
        timestamp_field="event_timestamp",
    ),
)

store = FeatureStore(repo_path=tmpdir)
store.apply([feature_window, ani_window_numeric_v1])

print("✅ Feast connected and feature view applied to ani-featurestore-snowflake")
print(f"   Online store : {FEAST_ONLINE_HOST}")
print(f"   Registry     : {FEAST_REGISTRY_HOST}")

## Step 5 — Push Features INTO Snowflake via the Online Server

The push goes to `ani-featurestore-snowflake` → it writes to Snowflake.  
In production this step is replaced by `feast materialize` from MinIO parquet files.

In [ ]:
import pandas as pd
from datetime import datetime, timezone

now = datetime.now(timezone.utc)
df  = pd.DataFrame([{
    "window_id":         DEMO_WINDOW_ID,
    "event_timestamp":   now,
    "created_timestamp": now,
    **features,
}])

print(f"Writing to ani-featurestore-snowflake online server...")
print(f"  window_id = '{DEMO_WINDOW_ID}'")
for k, v in features.items():
    print(f"  {k:<25} {v}")

store.write_to_online_store("ani_window_numeric_v1", df)

print(f"\n✅ Written to ani-featurestore-snowflake → Snowflake")

## Step 6 — Read Features FROM Snowflake via the Online Server

This is the same call the anomaly-service would make at inference time.

In [ ]:
import requests as req

print(f"Querying ani-featurestore-snowflake for window_id='{DEMO_WINDOW_ID}'...\n")

payload = {
    "features": [f"ani_window_numeric_v1:{f}" for f in NUMERIC_FEATURES],
    "entities": {"window_id": [DEMO_WINDOW_ID]},
}

resp = req.post(
    f"{FEAST_ONLINE_HOST}/get-online-features",
    json=payload,
    verify=CA_CERT,
    timeout=30,
)
resp.raise_for_status()
result = resp.json()

feature_names = result["metadata"]["feature_names"]
retrieved = {
    name: float(row["values"][0])
    for name, row in zip(feature_names, result["results"])
    if name != "window_id"
}

print("Features fetched FROM Snowflake (via ani-featurestore-snowflake online server):")
for name, value in retrieved.items():
    print(f"  {name:<25} {value}")

print("\n✅ Features retrieved from Snowflake online store")

## Step 7 — Send Snowflake Features to Triton for Prediction

In [ ]:
import requests

data = [[float(retrieved.get(f, 0.0)) for f in NUMERIC_FEATURES]]

print(f"Sending to Triton  : {TRITON_ENDPOINT}")
print(f"Model              : {MODEL_NAME}")
print(f"Input vector       : {data[0]}\n")

response = requests.post(
    f"{TRITON_ENDPOINT}/v2/models/{MODEL_NAME}/infer",
    json={
        "inputs": [{
            "name":     "predict",
            "shape":    [1, len(NUMERIC_FEATURES)],
            "datatype": "FP32",
            "data":     data,
        }]
    },
    timeout=30,
)
response.raise_for_status()
outputs = {o["name"]: o for o in response.json().get("outputs", [])}
print(f"✅ Triton responded with outputs: {list(outputs.keys())}")

## Step 8 — Results

In [ ]:
score_out = outputs.get("anomaly_score")
v = score_out["data"] if score_out else [[0.0]]
while isinstance(v, list):
    v = v[0]
anomaly_score = float(v)
is_anomaly    = anomaly_score > 0.5

print("=" * 55)
print("  INFERENCE RESULT")
print("=" * 55)
print(f"  Scenario           : {SCENARIO}")
print(f"  window_id          : {DEMO_WINDOW_ID}")
print(f"  Feature source     : ani-featurestore-snowflake")
print(f"  Online store       : Snowflake ANI_FEAST_DEMO.FEAST_ONLINE")
print(f"  anomaly_score      : {anomaly_score:.4f}")
print(f"  is_anomaly         : {is_anomaly}")
print("=" * 55)
print()
print("🚨 ANOMALY DETECTED" if is_anomaly else "✅ NORMAL TRAFFIC")
print()
print("Data flow:")
print("  Feast push → ani-featurestore-snowflake → Snowflake")
print("  get_online_features → ani-featurestore-snowflake → Snowflake")
print("  Triton → anomaly score")